# 06 — Sanity Check: Leakage, Score Plausibility, SHAP Plausibility
**ZivaBasa MVP (Kaggle-Data Phase)**

Before trusting any result from notebooks 03/04/05, run this. It automates the three checks that
actually matter on proxy data:

1. **Target distribution** — is each task's target sane (not degenerate, not all-one-class)?
2. **Leakage scan** — does any feature correlate suspiciously highly with its task's target?
3. **Score plausibility** — are baseline/NN scores in a believable range, or suspiciously perfect?
4. **SHAP plausibility** — do the top SHAP features make domain sense, and do they roughly agree
   with the tree-based feature importances from notebook 03?

This notebook doesn't fix anything — it flags. A flag means "look at this before you trust it,"
not automatically "something is wrong." On proxy Kaggle data, some flags are expected and fine
once you understand why (e.g. weak signal genuinely produces near-random scores).

**Input:** everything notebooks 02–05 already produced. Run this after 05, not standalone.


In [ ]:
# --- Setup ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

PROCESSED_DIR = "../data/processed"
MODELS_DIR = "../models"
SHAP_DIR = "../models/shap_outputs"

TASK_CONFIG = {
    "employment": {
        "target": "target_high_automation_risk",
        "task_type": "classification",
        "drop_cols": ["target_high_automation_risk", "automation_risk_score",
                      "automation_exposure_index", "exposure_x_skill_complexity"],
    },
    "skills": {
        "target": "target_attrition",
        "task_type": "classification",
        "drop_cols": ["target_attrition"],
    },
    "productivity": {
        "target": "target_ai_adoption",
        "task_type": "regression",
        "drop_cols": ["target_ai_adoption", "ai_adoption_level", "ai_adoption_index"],
    },
}

# Collects every flag raised across all sections, printed as one report at the end
flags = []  # list of dicts: {"section": str, "task": str, "severity": "warn"|"fail", "message": str}

def flag(section, task, severity, message):
    flags.append({"section": section, "task": task, "severity": severity, "message": message})
    icon = "🔴" if severity == "fail" else "🟡"
    print(f"{icon} [{section}/{task}] {message}")

def ok(section, task, message):
    print(f"🟢 [{section}/{task}] {message}")


## 1. Target Distribution Check

A degenerate target (e.g. 99% one class, or zero variance) makes every downstream metric
meaningless regardless of model quality. Check this first — it's the cheapest check and the one
most likely to explain a confusing result later.


In [ ]:
def load_features(name):
    path = os.path.join(PROCESSED_DIR, f"{name}_features.parquet")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run notebook 02 first.")
        return None
    return pd.read_parquet(path)

processed = {name: load_features(name) for name in TASK_CONFIG}

for name, cfg in TASK_CONFIG.items():
    df = processed[name]
    if df is None or cfg["target"] not in df.columns:
        flag("target_distribution", name, "fail", "Target column missing — nothing downstream can be trusted until this is fixed.")
        continue

    y = df[cfg["target"]]
    if cfg["task_type"] == "classification":
        counts = y.value_counts(normalize=True)
        minority_pct = counts.min() * 100
        print(f"[{name}] class balance:\n{(counts * 100).round(1)}")
        if minority_pct < 2:
            flag("target_distribution", name, "fail", f"Minority class is only {minority_pct:.1f}% — severe imbalance, metrics like accuracy will be misleading.")
        elif minority_pct < 10:
            flag("target_distribution", name, "warn", f"Minority class is {minority_pct:.1f}% — imbalanced; prefer ROC-AUC/F1 over accuracy, consider class_weight='balanced'.")
        else:
            ok("target_distribution", name, f"Class balance looks reasonable (minority class {minority_pct:.1f}%).")
    else:
        desc = y.describe()
        print(f"[{name}] target summary:\n{desc}")
        if desc["std"] < 1e-6:
            flag("target_distribution", name, "fail", "Target has ~zero variance — regression metrics are meaningless.")
        else:
            ok("target_distribution", name, f"Target has real variance (std={desc['std']:.4f}).")


## 2. Leakage Scan — Feature/Target Correlation

Flags any feature whose raw correlation with the target exceeds a threshold. This is the same
check that caught the `exposure_x_skill_complexity` leakage bug during development — it's cheap,
automatic, and catches the most common failure mode on engineered features.

Threshold is deliberately aggressive (0.3) since even "reasonable" engineered features on proxy
data shouldn't usually correlate that strongly with a threshold-derived or otherwise noisy target.
Adjust `CORR_THRESHOLD` if you have a specific reason to expect a strong legitimate driver.


In [ ]:
CORR_THRESHOLD = 0.30

for name, cfg in TASK_CONFIG.items():
    df = processed[name]
    if df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = df[cfg["target"]]

    corrs = X.corrwith(y).abs().sort_values(ascending=False)
    suspicious = corrs[corrs > CORR_THRESHOLD]

    print(f"\n[{name}] top 5 feature-target correlations:")
    print(corrs.head(5).round(3))

    if len(suspicious) > 0:
        for feat, corr_val in suspicious.items():
            flag("leakage_scan", name, "warn",
                 f"'{feat}' correlates {corr_val:.3f} with target — verify this isn't derived from the target's raw source column.")
    else:
        ok("leakage_scan", name, f"No feature exceeds |corr| > {CORR_THRESHOLD} with target.")


## 3. Score Plausibility — Baselines and Multi-Task NN

Flags scores that are **suspiciously high** (a leakage signal on proxy data — Section 9.2 of the
project documentation found a real bug exactly this way) as well as scores that are **exactly
chance level across every single model**, which usually means the target has no learnable signal
at all rather than a healthy "hard problem" result.


In [ ]:
SUSPICIOUS_HIGH = {"roc_auc": 0.93, "accuracy": 0.93, "r2": 0.85}
CHANCE_LEVEL = {"roc_auc": (0.45, 0.55)}

def load_csv_safe(path):
    if not os.path.exists(path):
        print(f"[MISSING] {path}")
        return None
    return pd.read_csv(path)

baseline_results = load_csv_safe(os.path.join(MODELS_DIR, "baseline_results.csv"))
nn_results = load_csv_safe(os.path.join(MODELS_DIR, "multitask_model", "multitask_nn_results.csv"))

all_results = pd.concat([r for r in [baseline_results, nn_results] if r is not None], ignore_index=True) \
    if (baseline_results is not None or nn_results is not None) else None

if all_results is not None:
    display(all_results)

    for task_name in TASK_CONFIG:
        task_rows = all_results[all_results["task_head"] == task_name]
        if task_rows.empty:
            continue

        task_type = TASK_CONFIG[task_name]["task_type"]
        metric_col = "roc_auc" if task_type == "classification" else "r2"
        if metric_col not in task_rows.columns:
            continue
        vals = task_rows[metric_col].dropna()
        if vals.empty:
            continue

        # Suspiciously high (possible leakage)
        threshold = SUSPICIOUS_HIGH.get(metric_col)
        if threshold and vals.max() > threshold:
            best_model = task_rows.loc[task_rows[metric_col].idxmax(), "model"]
            flag("score_plausibility", task_name, "warn",
                 f"{best_model} scores {metric_col}={vals.max():.3f} — unusually high for proxy data. "
                 f"Re-check Section 2 (leakage scan) and the feature dictionary before trusting this.")

        # All models stuck at chance level (classification only)
        if metric_col == "roc_auc":
            lo, hi = CHANCE_LEVEL["roc_auc"]
            if (vals > lo).sum() == 0 or (vals < hi).all():
                if vals.between(lo, hi).all():
                    flag("score_plausibility", task_name, "warn",
                         f"Every model is within chance-level ROC-AUC ({lo}-{hi}) — target may have no learnable "
                         f"signal from these features, or the target definition itself may need revisiting.")

        if not (threshold and vals.max() > threshold) and not (metric_col == "roc_auc" and vals.between(*CHANCE_LEVEL["roc_auc"]).all()):
            ok("score_plausibility", task_name, f"Score range for {metric_col} looks like a believable, non-trivial result.")
else:
    flag("score_plausibility", "all", "fail", "No baseline or NN results found — run notebooks 03 and 04 first.")


## 4. Baseline vs. Multi-Task NN — Does the Deep Model Earn Its Complexity?

Not a pass/fail flag, just the comparison itself — worth looking at directly rather than only
through the automated checks above.


In [ ]:
if all_results is not None:
    for task_name in TASK_CONFIG:
        task_rows = all_results[all_results["task_head"] == task_name]
        if task_rows.empty:
            continue
        task_type = TASK_CONFIG[task_name]["task_type"]
        metric_col = "roc_auc" if task_type == "classification" else "r2"
        if metric_col not in task_rows.columns or task_rows[metric_col].dropna().empty:
            continue

        sorted_rows = task_rows.sort_values(metric_col, ascending=False)
        best = sorted_rows.iloc[0]
        nn_row = task_rows[task_rows["model"] == "multitask_nn"]

        print(f"=== {task_name} ({metric_col}) ===")
        print(sorted_rows[["model", metric_col]].to_string(index=False))
        if not nn_row.empty and best["model"] != "multitask_nn":
            gap = best[metric_col] - nn_row.iloc[0][metric_col]
            print(f"  -> Best baseline beats multi-task NN by {gap:.3f} on {metric_col}. "
                  f"Worth noting explicitly in any write-up rather than only reporting the NN's number.")
        print()


## 5. SHAP Plausibility

Two checks: (a) do the top SHAP features look like things a domain expert would expect to matter,
and (b) do they roughly agree with the tree-based feature importances from notebook 03 — if SHAP
and Random Forest strongly disagree on what matters, that's worth understanding before either is
trusted for an HR-facing explanation.


In [ ]:
def load_shap_importance(name):
    path = os.path.join(SHAP_DIR, f"{name}_feature_importance.csv")
    if not os.path.exists(path):
        print(f"[MISSING] {path} — run notebook 05 first.")
        return None
    return pd.read_csv(path)

for name in TASK_CONFIG:
    shap_imp = load_shap_importance(name)
    if shap_imp is None:
        continue
    print(f"=== {name} — Top 10 SHAP Features ===")
    display(shap_imp.head(10))
    ok("shap_plausibility", name, "Top SHAP features printed above — review for domain sense manually; this can't be fully automated.")


In [ ]:
# Rank correlation between SHAP importance and Random Forest feature_importances_,
# where both are available. Low/negative correlation is a flag worth investigating, not
# necessarily a bug -- SHAP and impurity-based importance can legitimately diverge.
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

for name, cfg in TASK_CONFIG.items():
    shap_imp = load_shap_importance(name)
    df = processed[name]
    if shap_imp is None or df is None or cfg["target"] not in df.columns:
        continue

    drop_cols = [c for c in cfg["drop_cols"] if c in df.columns]
    X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = df[cfg["target"]]

    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42) \
        if cfg["task_type"] == "classification" else \
        RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
    rf.fit(X, y)
    rf_imp = pd.Series(rf.feature_importances_, index=X.columns, name="rf_importance")

    merged = shap_imp.set_index("feature")[["mean_abs_shap"]].join(rf_imp, how="inner")
    if len(merged) < 3:
        print(f"[{name}] too few overlapping features to compute rank correlation.")
        continue

    rho, pval = spearmanr(merged["mean_abs_shap"], merged["rf_importance"])
    print(f"[{name}] SHAP vs. Random Forest importance rank correlation: rho={rho:.3f} (p={pval:.3f})")
    if rho < 0.3:
        flag("shap_plausibility", name, "warn",
             f"SHAP and Random Forest importances only weakly agree (rho={rho:.3f}). "
             f"Not necessarily wrong, but worth understanding why before using SHAP output in an HR-facing explanation.")
    else:
        ok("shap_plausibility", name, f"SHAP and Random Forest importances broadly agree (rho={rho:.3f}).")


## 6. Summary Report


In [ ]:
print("=" * 70)
print("SANITY CHECK SUMMARY")
print("=" * 70)

if not flags:
    print("\n🟢 No flags raised. All checks passed cleanly.")
else:
    fails = [f for f in flags if f["severity"] == "fail"]
    warns = [f for f in flags if f["severity"] == "warn"]
    print(f"\n{len(fails)} FAIL, {len(warns)} WARN\n")
    if fails:
        print("🔴 FAIL (fix before trusting results):")
        for f in fails:
            print(f"   [{f['section']}/{f['task']}] {f['message']}")
    if warns:
        print("\n🟡 WARN (review, may be fine):")
        for f in warns:
            print(f"   [{f['section']}/{f['task']}] {f['message']}")

print("""

Remember throughout: this is proxy Kaggle data standing in for real banking-sector data.
A "clean" result here validates the PIPELINE, not real workforce findings. Keep that
distinction explicit in anything downstream of this notebook.
""")
